In [0]:
%python
from pyspark.sql.functions import *

spark.sql("USE CATALOG capstone")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

df = spark.table("capstone.bronze.diseases")

silver = (
    df
    # clean id
    .withColumn("disease_id", trim(col("disease_id")))

    # normalize names
    .withColumn("disease_name", initcap(trim(col("disease_name"))))
    .withColumn("disease_name_alt", initcap(trim(col("disease_name_alt"))))

    # normalize category
    .withColumn("category", initcap(trim(col("category"))))

    # severity cleanup
    .withColumn("severity_label", lower(trim(col("severity_label"))))

    # chronic flag
    .withColumn(
        "is_chronic",
        when(lower(col("chronic_flag_raw")).isin("yes","y","true","1"), True)
        .when(lower(col("chronic_flag_raw")).isin("no","n","false","0"), False)
    )

    # ICD cleanup
    .withColumn(
        "diagnosis_code",
        regexp_extract(
            upper(col("diagnosis_code_raw")),
            r"[A-Z][0-9]{2,3}",
            0
        )
    )

    # safe date parsing
    .withColumn("created_date", try_to_date("created_date_raw"))
    .withColumn("updated_date", try_to_date("updated_date_raw"))
)

final_df = silver.select(
    "disease_id",
    "disease_name",
    "disease_name_alt",
    "category",
    "description",
    "severity_label",
    "is_chronic",
    "diagnosis_code",
    "treatment_notes_raw",
    "status_raw",
    "created_date",
    "updated_date",
    "source_system",
    "source_record_id"
)

(
    final_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("capstone.silver.diseases")
)

print("Loaded capstone.silver.diseases")